In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import joblib

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()
rf = joblib.load('../backend/models/random_forest_model.pkl')

print("✅ Loaded")

✅ Loaded


In [2]:
# Cell 2 — Get probabilities for all test transactions
probs = rf.predict_proba(X_test)[:, 1]

score_df = pd.DataFrame({
    'actual': y_test,
    'fraud_prob': probs,
    'risk_score': (probs * 100).round(2)
})

print(score_df.head(10))

   actual  fraud_prob  risk_score
0       0        0.00         0.0
1       0        0.00         0.0
2       0        0.02         2.0
3       0        0.00         0.0
4       0        0.00         0.0
5       0        0.00         0.0
6       0        0.01         1.0
7       0        0.00         0.0
8       0        0.00         0.0
9       0        0.00         0.0


In [3]:
# Cell 3 — Assign risk levels
def get_risk_level(score):
    if score >= 70: 
        return "HIGH"
    elif score >= 40:
        return "MEDIUM"
    else:
        return "LOW"

score_df['risk_level'] = score_df['risk_score'].apply(get_risk_level)
print(score_df['risk_level'].value_counts())

risk_level
LOW       56859
HIGH         82
MEDIUM       21
Name: count, dtype: int64


In [4]:
# Cell 4 — How accurate is each risk level
for level in ['HIGH', 'MEDIUM', 'LOW']:
    subset = score_df[score_df['risk_level'] == level]
    fraud_rate = subset['actual'].mean() * 100
    print(f"{level}: {len(subset)} transactions | {fraud_rate:.2f}% actual fraud")

HIGH: 82 transactions | 93.90% actual fraud
MEDIUM: 21 transactions | 23.81% actual fraud
LOW: 56859 transactions | 0.03% actual fraud


In [5]:
# Cell 5 — Score a single transaction
def score_transaction(transaction, model):
    prob = model.predict_proba([transaction])[0][1]
    risk_score = round(prob * 100, 2)

    if risk_score >= 70:
        risk_level = "HIGH"
        emoji = "🔴"
    elif risk_score >= 40:
        risk_level = "MEDIUM"
        emoji = "🟡"
    else:
        risk_level = "LOW"
        emoji = "🟢"

    return {
        'fraud_probability': f"{round(prob * 100, 2)}%",
        'risk_score': risk_score,
        'risk_level': risk_level,
        'emoji': emoji
    }

# Test on a fraud transaction
fraud_idx = np.where(y_test == 1)[0][0]
transaction = X_test.iloc[fraud_idx].values

result = score_transaction(transaction, rf)
print("\n========== RISK SCORE ==========")
for k, v in result.items():
    print(f"  {k}: {v}")
print("================================")


========== RISK SCORE ==========
  fraud_probability: 94.0%
  risk_score: 94.0
  risk_level: HIGH
  emoji: 🔴


c:\Users\Constantine\Desktop\FraudLens\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
